In [1]:
%load_ext autoreload
%autoreload 2

import torch
from timm.models import create_model

model = create_model(
    'deit_small_distilled_patch16_224',
    pretrained=True,
    num_classes=1000,
    drop_path_rate=0.1,
    drop_block_rate=None,
    global_pool='token',
)

/home/chenzhiqiang/miniconda3/envs/det/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from compression import dyna_set_sparse_budget, replace_linear_with_, SparseLinear, create_compressed_deit_small

model = replace_linear_with_(
    model, 
    SparseLinear, 
    exclude=[model.get_classifier(), model.head, model.head_dist],
    N=2, M=4, hard=False)

import pickle

sparse_budget = pickle.load(open(
    '/home/chenzhiqiang/MaskLLM-4V/deit_small_layerwise_importance_20251223.pkl', 'rb'))

In [3]:
sd = torch.load('/home/chenzhiqiang/MaskLLM-4V/output/sparse_deit_small_patch16_224_90_epochs_lr1e-3_.augreg_in1k.hybird/HybirdSparse/model_best.pth.tar')

In [4]:
print(sd.keys())

dict_keys(['epoch', 'arch', 'state_dict', 'optimizer', 'version', 'args', 'amp_scaler', 'state_dict_ema', 'metric'])


In [5]:
print(sd['state_dict'].keys())

odict_keys(['cls_token', 'pos_embed', 'dist_token', 'patch_embed.proj.weight', 'patch_embed.proj.bias', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.qkv.gate', 'blocks.0.attn.qkv.mask', 'blocks.0.attn.proj.weight', 'blocks.0.attn.proj.bias', 'blocks.0.attn.proj.gate', 'blocks.0.attn.proj.mask', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'blocks.0.mlp.fc1.weight', 'blocks.0.mlp.fc1.bias', 'blocks.0.mlp.fc1.gate', 'blocks.0.mlp.fc1.mask', 'blocks.0.mlp.fc2.weight', 'blocks.0.mlp.fc2.bias', 'blocks.0.mlp.fc2.gate', 'blocks.0.mlp.fc2.mask', 'blocks.1.norm1.weight', 'blocks.1.norm1.bias', 'blocks.1.attn.qkv.weight', 'blocks.1.attn.qkv.bias', 'blocks.1.attn.qkv.gate', 'blocks.1.attn.qkv.mask', 'blocks.1.attn.proj.weight', 'blocks.1.attn.proj.bias', 'blocks.1.attn.proj.gate', 'blocks.1.attn.proj.mask', 'blocks.1.norm2.weight', 'blocks.1.norm2.bias', 'blocks.1.mlp.fc1.weight', 'blocks.1.mlp.fc1.bias', 'blocks.1.mlp.fc

In [6]:
model = create_compressed_deit_small(pretrained=False, n_bits=8, reduced_token=32, pt_state_dict=sd['state_dict'])

[SparseConfig] Model initialization complete


In [7]:
model

VisionTransformerDistilled(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): CompressedBlock(
      (norm1): IntLayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attn): CompressedAttention(
        (qkv): SparseQuantLinear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): SparseQuantLinear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
        (softmax): IntSoftmaxFixed()
        (matmul_qk): QuantMatmul(bits=8, init_state=0/8)
        (matmul_v): QuantMatmul(bits=8, init_state=0/8)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): IntLayerNorm((384,), e

In [8]:
temp = torch.ones((1, 3, 224, 224), device='cuda')
model = model.to('cuda')
o = model(temp)

In [9]:
print(model.blocks[0].mlp.fc1)

SparseQuantLinear(in_features=384, out_features=1536, bias=True)


: 

## Int LayerNorm

In [36]:
import torch.nn as nn
import math

class IntLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5, output_bit=8, quant_mode=True, force_dequant="none"):
        super().__init__()
        self.D = normalized_shape[0] if isinstance(normalized_shape, (list, tuple)) else normalized_shape
        self.eps = eps
        
        self.weight = nn.Parameter(torch.ones(self.D))
        self.bias = nn.Parameter(torch.zeros(self.D))

        self.quant_mode = quant_mode
        if force_dequant in ["nonlinear", "layernorm"]:
            self.quant_mode = False

        self.register_buffer("shift", torch.tensor([0], dtype=torch.long))
        self.output_bit = output_bit
        self.max_bit = 32 # Hardware accumulator limit
        
        # --- Hardware Approximation Configurations ---
        self.Q_D = 16
        self.inv_D = int(round((1 << self.Q_D) / self.D))
        print("self.inv_D:", self.inv_D)
        
        self.m = 4        # Number of mantissa bits for LUT index
        # self.Q_lut = 15   # Fractional bits for LUT values
        self.Q_lut = 8   # Fractional bits for LUT values
        self._init_isqrt_lut()

        self.Q_w = 8      # Affine transform precision
        
    def _init_isqrt_lut(self):
        """
        Pre-computes the Inverse Square Root LUT.
        Size = 2^(m+1). Index is formed by: [parity of k (1 bit)] | [mantissa (m bits)]
        """
        lut_size = 1 << (self.m + 1)
        lut = torch.zeros(lut_size, dtype=torch.long)
        for i in range(lut_size):
            parity = i >> self.m
            mantissa = i & ((1 << self.m) - 1)
            # Reconstruct the normalized value: 2^parity * (1 + mantissa / 2^m)
            val = (2 ** parity) * (1.0 + mantissa / (1 << self.m))
            # Store scaled ISqrt value
            lut[i] = int(round((1 << self.Q_lut) / math.sqrt(val)))
            print(f"LUT{i}:", lut[i])
        self.register_buffer("isqrt_lut", lut)

    def set_shift(self, y_int):
        """ Adjusts dynamic shift using L_infinity norm to mathematically guarantee no overflow """
        with torch.no_grad():
            y_max = y_int.abs().max().float()
            max_val_allowed = math.sqrt(((1 << self.max_bit) - 1) / self.D)
            
            if y_max > max_val_allowed:
                # Calculate required shift to keep max squared sum strictly within max_bit
                required_shift = torch.ceil(torch.log2(y_max / max_val_allowed)).long()
                self.shift = torch.max(self.shift, required_shift)

    def forward(self, x_int, scaling_factor):
        # if not self.quant_mode:
        #     mean = x.mean(dim=2, keepdim=True)
        #     y = x - mean
        #     var = torch.mean(y**2, dim=2, keepdim=True)
        #     return (y / torch.sqrt(self.eps + var)) * self.weight + self.bias, None

        # --- Hardware-Equivalent Integer Forward Pass ---
        
        print("Step 1 Input quantized:", x_int[0][0][:8])

        # [Step 1] Division-free Mean Calculation
        sum_x = x_int.sum(dim=2, keepdim=True)
        mean_int = torch.bitwise_right_shift(sum_x * self.inv_D, self.Q_D)
        print("Step 2 sum & mean:", sum_x, mean_int)
        
        y_int = x_int - mean_int
        print("Step 3 shifted:", y_int[0][0][:8])

        # [Step 2] Variance Calculation with Dynamic Overflow Prevention
        y_shifted = torch.bitwise_right_shift(y_int, self.shift.item())
        y_sq_int = y_shifted * y_shifted
        sum_sq = y_sq_int.sum(dim=2, keepdim=True)
        print("Step 4 sum of square:", sum_sq)
        
        # Check accumulator overflow on SUM OF SQUARES, not variance!
        if self.training and sum_sq.max() >= (1 << self.max_bit):
            self.set_shift(y_int)
            y_shifted = torch.bitwise_right_shift(y_int, self.shift.item())
            sum_sq = (y_shifted * y_shifted).sum(dim=2, keepdim=True)
            
        var_int = torch.bitwise_right_shift(sum_sq * self.inv_D, self.Q_D)
        var_int = torch.clamp(var_int, min=1) # Prevents log2(0)
        print("Step 5 var_int:", var_int)

        # [Step 3] LOD & LUT-based Inverse Square Root
        k = var_int.float().log2().long() 
        parity = k & 1
        mask = (1 << self.m) - 1
        
        shift_amt = k - self.m
        mantissa = torch.where(
            shift_amt >= 0,
            torch.bitwise_right_shift(var_int, shift_amt) & mask,
            torch.bitwise_left_shift(var_int, -shift_amt) & mask
        )
        
        idx = (parity << self.m) | mantissa
        inv_sqrt_val = self.isqrt_lut[idx]
        p = torch.bitwise_right_shift(k, 1)
        
        total_shift = p + self.shift.item() + self.Q_lut
        print(f"k, parity, mantissa, idx, inv_sqrt_val, shift:", k, parity, mantissa, idx, inv_sqrt_val, total_shift)
        
        # Quantize Weights and Bias 
        weight_int = (self.weight * (1 << self.Q_w)).round().long()
        bias_int = (self.bias * (1 << self.Q_w)).round().long() 
        
        # Fusion: Y = Y_int * ISqrt * W
        scaled_y = y_int * inv_sqrt_val * weight_int 
        
        # Bit-shift to align with Q_w fractional bits, then add appropriately scaled Bias
        # Broadcasting caveat: total_shift is [B, S, 1], out_int becomes [B, S, D]
        out_int = torch.bitwise_right_shift(scaled_y, total_shift) + bias_int
        
        # Thus out_int purely represents True_FP32_Output * 2^Q_w.
        out_fp32 = out_int.float() / (1 << self.Q_w)

        # Output logic for downstream integer layers:
        # If downstream expects an integer tensor with scale `scaling_factor`:
        new_scaling_factor = scaling_factor if scaling_factor is not None else 1.0
        
        # For seamless integration back to floating point PyTorch framework:
        return out_fp32, new_scaling_factor

In [37]:
import torch
import torch.nn as nn 

def evaluate_layernorm_approximation():
    torch.manual_seed(42)
    
    # 模拟输入参数
    batch_size, seq_len, dim = 1, 1, 384
    input_scale = 0.03 # 模拟前面网络层的浮点缩放因子
    
    # 生成随机浮点输入
    x_fp = torch.randn(batch_size, seq_len, dim) * 2.0 + 0.5
    # print("Generated Random Val:", x_fp[0][0][:8])
    print("Generated Random Val:", x_fp[0][0])
    
    # 模拟硬件输入的 8-bit 量化整型
    x_int = (x_fp / input_scale).round().long().clamp(-128, 127)
    
    # 重新算回作为 Standard FP32 的实际输入 (消除初始输入量化误差，专注于 LN 内部的误差)
    x_fp_actual = x_int.float() * input_scale
    
    # 1. 初始化标准 LayerNorm
    std_ln = nn.LayerNorm(dim, eps=0.0)
    # 给予一些随机权重和偏置测试鲁棒性
    nn.init.uniform_(std_ln.weight, 0.8, 1.2)
    nn.init.uniform_(std_ln.bias, -0.1, 0.1)
    
    # 2. 初始化整形近似 LayerNorm，并对齐权重
    int_ln = IntLayerNorm(dim)
    int_ln.weight.data.copy_(std_ln.weight.data)
    int_ln.bias.data.copy_(std_ln.bias.data)
    
    # 校准一次以获取动态 Shift 值
    int_ln.set_shift(x_int)
    print(f"[*] Calibration finished. Dynamic Shift determined: {int_ln.shift.item()}")

    # 3. 前向传播
    with torch.no_grad():
        out_std = std_ln(x_fp_actual)
        print("out_std", out_std[0][0][0:10])
        out_int, out_int_scale = int_ln(x_int, input_scale)
        print("out_int:", out_int[0][0][0:10])
        
    # 4. 误差度量
    mse = torch.nn.functional.mse_loss(out_std, out_int).item()
    max_err = torch.max(torch.abs(out_std - out_int)).item()
    
    # 余弦相似度 (衡量向量方向)
    cos_sim = torch.nn.functional.cosine_similarity(
        out_std.view(-1, dim), 
        out_int.view(-1, dim), 
        dim=-1
    ).mean().item()
    
    print("\n" + "="*40)
    print("LayerNorm 整形查表近似精度评估报告")
    print("="*40)
    print(f"Mean Squared Error (MSE):  {mse:.6f}")
    print(f"Max Absolute Error:        {max_err:.6f}")
    print(f"Cosine Similarity:         {cos_sim:.6f} (理想值为 1.0)")
    print("="*40)

evaluate_layernorm_approximation()

Generated Random Val: tensor([ 4.3538e+00,  3.4746e+00,  2.3014e+00, -3.7110e+00,  1.8568e+00,
        -1.9691e+00,  4.1387e-01, -2.7093e+00, -1.0043e+00,  3.7974e+00,
        -2.8496e-01, -2.3072e+00, -9.5576e-01, -6.1886e-01, -1.0377e+00,
         2.0249e+00,  3.7846e+00,  1.8081e-01, -4.9480e-01,  1.3792e+00,
        -1.0163e+00,  2.6566e+00,  2.1016e+00,  3.8612e+00,  3.0582e+00,
         3.0928e+00,  1.7209e+00,  3.1695e+00,  3.6751e-02,  5.8352e-01,
        -3.1506e-03,  2.2197e+00, -2.2693e+00, -1.2425e+00,  5.3268e-02,
         3.9347e+00,  1.1378e+00, -3.4904e-01,  1.1114e+00, -1.0492e+00,
        -2.6151e+00,  2.4913e+00, -1.2596e+00, -7.0228e-01, -2.0483e+00,
         4.7456e+00, -1.9693e+00, -4.7583e-01, -1.3276e+00, -8.1627e-01,
         6.5605e-01,  1.5516e+00, -4.7598e-01,  2.8827e+00, -1.1280e+00,
        -9.7199e-01, -2.3065e+00,  5.7201e-01,  3.7305e-01,  1.8512e+00,
         3.0439e-01,  4.1892e+00, -1.8691e+00,  3.2671e+00,  3.3903e+00,
         2.2128e+00,  4.9362e

## Visualization of Quant scale & Requant params

In [2]:
import torch

state_dict = torch.load('/home/chenzhiqiang/SparseQuant/output/step2_sparse_deit_small_patch16_224_30_epochs_lr1e-3_tp32_8bits.augreg_in1k.hybird/HybirdSparse/last.pth.tar')

In [5]:
print(state_dict['state_dict'].keys())
cur_state_dict = state_dict['state_dict']
qkv_w_alpha = cur_state_dict['blocks.0.attn.qkv.weight_alpha']
qkv_a_alpha = cur_state_dict['blocks.0.attn.qkv.act_alpha']
proj_a_alpha = cur_state_dict['blocks.0.attn.matmul_qk.a_alpha']

rescale = proj_a_alpha / (qkv_w_alpha * qkv_a_alpha)

odict_keys(['cls_token', 'pos_embed', 'dist_token', 'patch_embed.proj.weight', 'patch_embed.proj.bias', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.norm1.norm_scaling_factor', 'blocks.0.norm1.bias_integer', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.qkv.weight_alpha', 'blocks.0.attn.qkv.act_alpha', 'blocks.0.attn.qkv.init_state', 'blocks.0.attn.qkv.mask', 'blocks.0.attn.proj.weight', 'blocks.0.attn.proj.bias', 'blocks.0.attn.proj.weight_alpha', 'blocks.0.attn.proj.act_alpha', 'blocks.0.attn.proj.init_state', 'blocks.0.attn.proj.mask', 'blocks.0.attn.matmul_qk.a_alpha', 'blocks.0.attn.matmul_qk.b_alpha', 'blocks.0.attn.matmul_qk.init_state', 'blocks.0.attn.matmul_v.a_alpha', 'blocks.0.attn.matmul_v.b_alpha', 'blocks.0.attn.matmul_v.init_state', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'blocks.0.norm2.norm_scaling_factor', 'blocks.0.norm2.bias_integer', 'blocks.0.mlp.fc1.weight', 'blocks.0.mlp.fc1.bias', 'blocks.0.mlp.fc1.weight_alpha', 'blo

In [1]:
import torch
import torch.nn as nn
import timm

def convert_conv_to_matmul(model, x):
    """
    Convert the first convolutional layer (patch embedding) of a DeiT-small model
    to matrix multiplication form (im2col + GEMM) and verify equivalence.
    Also prints the matrix operation dimensions, parameter count, and FLOPs.
    """
    # Extract the first convolution layer (patch_embed.proj)
    conv_layer = model.patch_embed.proj  # Conv2d(3, 384, kernel_size=16, stride=16)
    print(f"Original convolution layer: {conv_layer}")

    # Original convolution output (used as reference)
    with torch.no_grad():
        y_conv = conv_layer(x)

    # ----- Dimensions and parameters -----
    B, C, H, W = x.shape
    out_ch = conv_layer.out_channels
    k = conv_layer.kernel_size[0]          # kernel size (square)
    s = conv_layer.stride[0]               # stride
    out_h = (H - k) // s + 1               # 14
    out_w = (W - k) // s + 1               # 14
    N = out_h * out_w                      # number of patches (196)
    D = C * k * k                          # unfolded feature dimension (3*16*16=768)

    print("\n========== Matrix Operation Scale ==========")
    print(f"Input batch size (B): {B}")
    print(f"Input channels (C): {C}, height (H): {H}, width (W): {W}")
    print(f"Kernel size: {k}x{k}, stride: {s}")
    print(f"Output feature map size: {out_h}x{out_w} (total patches N = {N})")
    print(f"Unfolded dimension per patch (D = C*k*k): {D}")
    print(f"Output channels (out_ch): {out_ch}")
    print(f"Weight matrix W shape: ({out_ch}, {D})")
    print(f"Patch matrix X shape: (B, {D}, {N})")
    print(f"Matrix multiplication Y = W * X  ->  shape: (B, {out_ch}, {N})")
    print(f"Reshaped to spatial: (B, {out_ch}, {out_h}, {out_w})")

    # ----- Parameter count -----
    conv_params = out_ch * D
    if conv_layer.bias is not None:
        conv_params += out_ch
    print(f"\n========== Parameter Count ==========")
    print(f"Convolution / weight matrix parameters: {out_ch} * {D} = {out_ch * D:,}")
    print(f"Bias parameters: {out_ch if conv_layer.bias is not None else 0}")
    print(f"Total parameters in this layer: {conv_params:,}")

    # ----- Compute FLOPs (Multiply-Add operations) -----
    # Each output element requires D multiplications and (D-1) additions.
    # Total multiply-add operations = B * out_ch * N * (2*D - 1)
    # But we can also count multiplications and additions separately.
    mults = B * out_ch * N * D
    adds = B * out_ch * N * (D - 1)
    total_ops = mults + adds   # FLOPs (floating point operations)
    # Usually one multiply-add (MAC) counts as 2 FLOPs, but here we count each op.
    print(f"\n========== Computational Cost (FLOPs) ==========")
    print(f"Matrix multiplication dimensions: ({out_ch}×{D}) · ({D}×{N}) per batch")
    print(f"Multiplications: {mults:,}")
    print(f"Additions:       {adds:,}")
    print(f"Total FLOPs:     {total_ops:,}  ( = mults + adds )")
    print(f"Equivalent to convolution FLOPs: {total_ops:,} (identical by design)")

    # ----- Convert to matrix multiplication (im2col + GEMM) -----
    # 1. Unfold input image into patches (im2col)
    patches = torch.nn.functional.unfold(x, kernel_size=k, stride=s)  # (B, D, N)

    # 2. Flatten convolution weights: (out_ch, C, k, k) -> (out_ch, D)
    weight = conv_layer.weight                     # shape: (out_ch, C, k, k)
    w_flat = weight.view(weight.size(0), -1)       # shape: (out_ch, D)

    # 3. Perform matrix multiplication
    y_matmul = torch.matmul(w_flat, patches)       # (B, out_ch, N)

    # 4. Reshape to spatial feature map: (B, out_ch, out_h, out_w)
    y_matmul = y_matmul.view(B, out_ch, out_h, out_w)

    # 5. Add bias if present
    if conv_layer.bias is not None:
        y_matmul += conv_layer.bias.view(1, -1, 1, 1)

    # ----- Verification -----
    diff = torch.abs(y_conv - y_matmul).max().item()
    print(f"\n========== Verification ==========")
    print(f"Maximum absolute difference between original conv and matrix multiplication: {diff:.6e}")
    assert diff < 1e-5, "Matrix multiplication result does not match convolution!"
    print("✅ Verification passed: matrix multiplication equivalent to convolution.\n")

    return y_matmul

if __name__ == "__main__":
    # Create DeiT-small model (pretrained=False for random weights)
    model = timm.create_model('deit_small_patch16_224', pretrained=False)
    model.eval()

    # Generate random input image (batch size = 2, 3 channels, 224x224)
    batch_size = 2
    x = torch.randn(batch_size, 3, 224, 224)

    # Run conversion and verification
    y = convert_conv_to_matmul(model, x)
    print(f"Output shape: {y.shape}")  # Expected: (2, 384, 14, 14)

/home/chenzhiqiang/miniconda3/envs/det/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Original convolution layer: Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))

========== Matrix Operation Scale ==========
Input batch size (B): 2
Input channels (C): 3, height (H): 224, width (W): 224
Kernel size: 16x16, stride: 16
Output feature map size: 14x14 (total patches N = 196)
Unfolded dimension per patch (D = C*k*k): 768
Output channels (out_ch): 384
Weight matrix W shape: (384, 768)
Patch matrix X shape: (B, 768, 196)
Matrix multiplication Y = W * X  ->  shape: (B, 384, 196)
Reshaped to spatial: (B, 384, 14, 14)

========== Parameter Count ==========
Convolution / weight matrix parameters: 384 * 768 = 294,912
Bias parameters: 384
Total parameters in this layer: 295,296

========== Computational Cost (FLOPs) ==========
Matrix multiplication dimensions: (384×768) · (768×196) per batch
Multiplications: 115,605,504
Additions:       115,454,976
Total FLOPs:     231,060,480  ( = mults + adds )
Equivalent to convolution FLOPs: 231,060,480 (identical by design)

========== Ver